# Validação cruzada do classificador

Agrega os 5 modelos treinados por `src/classification/cross_validate.py` (um por fold, ver `src/preprocessing/pipeline.py::generate_kfold_splits`). Cada modelo treinou em 4 folds, usou o quinto como validação (early stopping e escolha do checkpoint) e foi avaliado no **mesmo split de teste fixo** — o desvio padrão entre folds mede quanto o resultado depende da partição de treino.

Rodar na máquina de treino (dentro do tmux), antes deste notebook:

```bash
PYTHONUNBUFFERED=1 uv run --group classifier python src/classification/cross_validate.py --run-name resnet50-paper-cv | tee models/classifier/runs/resnet50-paper-cv.log
```

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src" / "classification"))
sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from metrics import SCALAR_METRICS, classification_metrics, summarize_folds
from voc_to_yolo import DENVER_CLASSES

RUNS = REPO_ROOT / "models" / "classifier" / "runs"
CV_RUN = RUNS / "resnet50-paper-cv"
SINGLE_RUN = RUNS / "resnet50-paper"  # split único 70/15/15, para comparação
FIGURES = REPO_ROOT / "outputs" / "figures" / "classifier"
FIGURES.mkdir(parents=True, exist_ok=True)

METRIC_NAMES = {
    "accuracy": "Acurácia", "macro_precision": "Precisão (macro)", "macro_recall": "Recall (macro)",
    "macro_f1": "F1 (macro)", "macro_auc": "AUC-ROC (macro, OvR)",
}

Métricas de teste de cada fold, recalculadas a partir das probabilidades salvas (`fold*/test_probs.npy`).

In [ ]:
fold_dirs = sorted(d for d in CV_RUN.glob("fold*") if (d / "test_probs.npy").exists())
print(f"Folds avaliados: {[d.name for d in fold_dirs]}")

fold_metrics = {
    d.name: classification_metrics(np.load(d / "test_probs.npy"), np.load(d / "test_labels.npy"))
    for d in fold_dirs
}
per_fold = pd.DataFrame(
    {name: {METRIC_NAMES[k]: m[k] for k in SCALAR_METRICS} for name, m in fold_metrics.items()}
).T
per_fold.round(4)

Resumo em **média ± desvio padrão** (desvio amostral entre os folds), ao lado do modelo do split único (`resnet50-paper`) avaliado no mesmo teste.

In [ ]:
summary = summarize_folds(list(fold_metrics.values()))
single = classification_metrics(np.load(SINGLE_RUN / "test_probs.npy"), np.load(SINGLE_RUN / "test_labels.npy"))

table = pd.DataFrame(
    {
        "Validação cruzada (5 folds)": {METRIC_NAMES[k]: f"{m:.4f} ± {s:.4f}" for k, (m, s) in summary.items()},
        "Split único (resnet50-paper)": {METRIC_NAMES[k]: f"{single[k]:.4f}" for k in SCALAR_METRICS},
    }
)
table.to_csv(FIGURES / "cv_summary.csv")
table

F1 por classe (média ± desvio entre folds). As classes com maior desvio são as mais sensíveis à partição de treino — o Y, a mais rara, é a candidata óbvia.

In [ ]:
f1 = np.stack([m["per_class"]["f1"] for m in fold_metrics.values()])
per_class = pd.DataFrame(
    {"F1 médio": f1.mean(axis=0), "desvio": f1.std(axis=0, ddof=1)}, index=DENVER_CLASSES
)
per_class.to_csv(FIGURES / "cv_per_class_f1.csv")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(per_class.index, per_class["F1 médio"], yerr=per_class["desvio"], capsize=3, color="#4C72B0")
ax.set_ylim(max(0.0, per_class["F1 médio"].min() - 0.05), 1.0)
ax.set_ylabel("F1 no teste")
ax.set_title("F1 por classe — validação cruzada (média ± desvio, 5 folds)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES / "cv_per_class_f1.png", dpi=200)
plt.show()

per_class.sort_values("desvio", ascending=False).head(5).round(4)

Curvas de validação de cada fold (`history.csv`): confere se os folds convergem de forma parecida e em quantas épocas o early stopping parou.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for d in fold_dirs:
    history = pd.read_csv(d / "history.csv")
    axes[0].plot(history["epoch"], history["val_loss"], label=d.name)
    axes[1].plot(history["epoch"], history["val_f1"], label=d.name)
axes[0].set_title("Loss de validação")
axes[1].set_title("F1 macro de validação")
for ax in axes:
    ax.set_xlabel("época")
    ax.grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURES / "cv_history.png", dpi=200)
plt.show()